<a href="https://colab.research.google.com/github/rahman1487/Chatbot-Analytics-and-Optimization/blob/main/ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y transformers peft trl accelerate sentence-transformers

!pip install -q \
transformers==4.41.2 \
accelerate==0.30.1 \
datasets==2.19.2 \
evaluate==0.4.2 \
scikit-learn \
pandas \
numpy \
matplotlib \
seaborn \
openpyxl

In [ ]:
import transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

In [ ]:
from transformers import Trainer
from transformers import TrainingArguments

print("Trainer Loaded Successfully")

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

In [ ]:
!ls /content

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
print(train_df.head())
print(train_df.shape)
print(test_df.shape)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_df["text"] = (
    train_df["Title"].astype(str)
    + " "
    + train_df["Description"].astype(str)
)

test_df["text"] = (
    test_df["Title"].astype(str)
    + " "
    + test_df["Description"].astype(str)
)

train_df["label"] = train_df["Class Index"] - 1
test_df["label"] = test_df["Class Index"] - 1

train_df = train_df[["text","label"]]
test_df = test_df[["text","label"]]

print(train_df.head())

In [ ]:
train_df = train_df.sample(
    n=5000,
    random_state=42
)

test_df = test_df.sample(
    n=1000,
    random_state=42
)

print(train_df.shape)
print(test_df.shape)

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
results = []

confusion_matrices = {}

def train_model(model_name):

    print(f"\nTraining {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    def tokenize(batch):

        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=64
        )

    train_tok = train_dataset.map(
        tokenize,
        batched=True
    )

    test_tok = test_dataset.map(
        tokenize,
        batched=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=4
    )

    training_args = TrainingArguments(
        output_dir=f"./{model_name}",
        num_train_epochs=1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        save_strategy="no",
        logging_steps=100,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=test_tok,
        compute_metrics=compute_metrics
    )

    start_time = time.time()

    trainer.train()

    training_time = time.time() - start_time

    predictions = trainer.predict(
        test_tok
    )

    y_true = predictions.label_ids

    y_pred = np.argmax(
        predictions.predictions,
        axis=1
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted"
    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    confusion_matrices[model_name] = cm

    results.append([
        model_name,
        accuracy,
        precision,
        recall,
        f1,
        training_time
    ])

In [ ]:
import requests

r = requests.get(
    "https://huggingface.co/bert-base-uncased/raw/main/config.json"
)

print(r.status_code)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

print("Tokenizer Loaded")

In [ ]:
train_model(
    "bert-base-uncased"
)


Training bert-base-uncased


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Step,Training Loss


In [ ]:
train_model(
    "roberta-base"
)

In [ ]:
train_model(
    "distilbert-base-uncased"
)

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Training Time"
    ]
)

results_df

In [ ]:
results_df.to_csv(
    "Transformer_Results.csv",
    index=False
)

results_df

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Accuracy"]
)

plt.title(
    "Accuracy Comparison"
)

plt.ylabel(
    "Accuracy"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Precision"]
)

plt.title(
    "Precision Comparison"
)

plt.ylabel(
    "Precision"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Recall"]
)

plt.title(
    "Recall Comparison"
)

plt.ylabel(
    "Recall"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["F1 Score"]
)

plt.title(
    "F1 Score Comparison"
)

plt.ylabel(
    "F1 Score"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Training Time"]
)

plt.title(
    "Training Time Comparison"
)

plt.ylabel(
    "Seconds"
)

plt.show()

In [ ]:
for model_name, cm in confusion_matrices.items():

    plt.figure(figsize=(6,5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues"
    )

    plt.title(
        f"{model_name} Confusion Matrix"
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "Actual"
    )

    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.style.use('ggplot')

df = pd.read_excel("Untitled form (Responses).xlsx")

questions = df.columns[1:]

chart_styles = [
    "pie",
    "bar",
    "barh",
    "donut",
    "line",
    "area",
    "pie",
    "bar",
    "barh",
    "donut",
    "line",
    "area",
    "pie",
    "bar",
    "donut"
]

color_palettes = [
    "Set3",
    "viridis",
    "magma",
    "Pastel1",
    "coolwarm",
    "crest",
    "Accent",
    "rocket",
    "tab20",
    "Pastel2",
    "flare",
    "cubehelix",
    "Dark2",
    "husl",
    "Set2"
]

for i, col in enumerate(questions):

    print("\n" + "="*80)
    print(f"Question {i+1}: {col}")
    print("="*80)

    freq = df[col].value_counts()

    percent = round(
        (freq / freq.sum()) * 100,
        2
    )

    summary = pd.DataFrame({
        "Frequency": freq,
        "Percentage": percent
    })

    print(summary)

    colors = sns.color_palette(
        color_palettes[i],
        len(freq)
    )

    plt.figure(figsize=(9,6))

    chart_type = chart_styles[i]

    if chart_type == "pie":

        plt.pie(
            freq,
            labels=freq.index,
            autopct="%1.1f%%",
            colors=colors,
            startangle=90
        )

        plt.title(col)

    elif chart_type == "donut":

        plt.pie(
            freq,
            labels=freq.index,
            autopct="%1.1f%%",
            colors=colors,
            wedgeprops={"width":0.4}
        )

        plt.title(col)

    elif chart_type == "bar":

        sns.barplot(
            x=freq.index,
            y=freq.values,
            palette=color_palettes[i]
        )

        plt.xticks(rotation=30)
        plt.ylabel("Frequency")
        plt.title(col)

    elif chart_type == "barh":

        sns.barplot(
            y=freq.index,
            x=freq.values,
            palette=color_palettes[i]
        )

        plt.xlabel("Frequency")
        plt.title(col)

    elif chart_type == "line":

        plt.plot(
            freq.index,
            freq.values,
            marker='o',
            linewidth=3
        )

        plt.xticks(rotation=30)
        plt.ylabel("Frequency")
        plt.title(col)

    elif chart_type == "area":

        plt.fill_between(
            range(len(freq)),
            freq.values,
            alpha=0.6
        )

        plt.xticks(
            range(len(freq)),
            freq.index,
            rotation=30
        )

        plt.ylabel("Frequency")
        plt.title(col)

    plt.tight_layout()

    plt.show()